# 03 · Исправленная калибровка, soft voting и test · v2
## tl;dr
По развитию модели выбран **C_ovr/single**. После фиксации правил исследовательский повторный test: macro-F1 **0.8538**, recall патологии **91.4%**, AP патологии **0.9103**. Архитектура/калибровка выбирались по групповой validation, не по test. Ансамбль усредняет вероятности классов; старая ошибка перемножения средних исправлена в новой реализации.

## Context & Methods

### Key Assumptions

- Деревья fit только на train. Для каждого семейства новая полная 3-fold сетка из 24 комбинаций (логистика 6). Финальная калибровка использует отдельную validation и group cross-fitting для выбора метода. Деревья после калибровки на validation не переобучаются.
- Для shortlist дополнительно обучаются пять моделей на 4/5 train. Усредняются **три вероятности классов каждой модели**. Для иерархии сначала строятся [1−a,a(1−b),ab], затем среднее: E[ab] отличается от E[a]E[b]. Калибровка отдельно обучается на выходе именно этого среднего.
- Калибровка выбирается по log loss, решение порога — по cross-fitted macro-F1 с минимальным выигрышем .005. Ансамбль принимается только при приросте validation macro-F1 ≥.01. Обе границы зафиксированы до нового test.
- В validation лишь семь исходных файлов с патологией. Даже баланс независимых файлов не позволяет разделить большой файл на независимые наблюдения. Выбор postprocessing может быть шумным; улучшение на validation не гарантирует улучшение test.
- Внутренние cross-fit числа служат подбору: базовые OOF-прогнозы и новые calibration-фолды не дают полностью независимой meta-CV внутри fit. Кроме того, holdout-порог fit по вероятностям своего обученного calibrator, а итоговый cutoff — по cross-fitted probabilities. Поэтому validation-оценка архитектуры приблизительна; внешняя CV02 независимо оценивает фактически возвращаемую процедуру. Эти числа не называются selection-adjusted оценкой уже выбранного среди шести семейств победителя.
- Файл решения сохраняется до test predict. Ранее известный test используется для исследовательской перепроверки, не для выбора семейства, параметров или архитектуры. Интервалы на test — paired bootstrap исходных файлов, условные на фиксированных моделях, без разброса переобучения.

In [3]:
from pathlib import Path
import sys,json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.training.ctg_audit import (AUDIT,EXPORT,context,CTG_FEATURES,score,soft_vote)
data,fingerprint,cache=context()
sns.set_theme(style='whitegrid',font_scale=1.05)
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'figure.figsize':(10,5)})
def show_save(name):
    plt.gcf().savefig(AUDIT/name,dpi=140,bbox_inches='tight')
    plt.show()

## Data

In [5]:
selection=json.loads((AUDIT/'selection.json').read_text())
print('Primary family:',selection['primary']);print('Shortlist:',selection['shortlist'])
display(data.groupby('split').agg(segments=('source_file','size'),recordings=('source_file','nunique')))

Primary family: C_ovr
Shortlist: ['C_ovr', 'D_hierarchy']


,segments,recordings
split,,
test,425,68
train,1360,227
validation,341,57


## Results · финальный поиск и фиксация

In [7]:
from src.training.ctg_audit import run_final
results,locked=run_final()
print('Primary:',locked['primary']);print('Exported shortlist:',locked['shortlist'])

Audited final A_multiclass: complete
Audited final B_sqrt: complete
Audited final B_balanced: complete
Audited final C_ovr: complete
Audited final D_hierarchy: complete
Audited final logistic: complete
Primary: C_ovr/single
Exported shortlist: ['C_ovr/single', 'D_hierarchy/single']


### Validation: поддержка и правила

In [9]:
display(pd.read_csv(AUDIT/'validation_fold_support.csv'))
v=pd.read_csv(AUDIT/'validation_policy_metrics.csv')
display(v[['candidate','calibration','decision','macro_f1','recall_path','log_loss']].round(4))
display(pd.DataFrame(locked['policies']).T)

,fold,n_fit,n_hold,source_files,normal_segments,normal_files,suspect_segments,suspect_files,path_segments,path_files
0,0,230,111,19,90,14,15,7,6,2
1,1,230,111,19,89,14,16,6,6,3
2,2,222,119,19,87,15,15,6,17,2


,candidate,calibration,decision,macro_f1,recall_path,log_loss
0,A_multiclass/single,raw,argmax,0.8139,0.8621,0.2288
1,A_multiclass/single,raw,threshold,0.8115,0.8966,0.2288
2,A_multiclass/single,temperature,argmax,0.8139,0.8621,0.2306
3,A_multiclass/single,temperature,threshold,0.8115,0.8966,0.2306
4,A_multiclass/single,sigmoid,argmax,0.8300,0.8276,0.2300
5,A_multiclass/single,sigmoid,threshold,0.8140,0.8966,0.2300
6,B_sqrt/single,raw,argmax,0.8332,0.8966,0.2532
7,B_sqrt/single,raw,threshold,0.8243,0.8621,0.2532
8,B_sqrt/single,temperature,argmax,0.8332,0.8966,0.2539
9,B_sqrt/single,temperature,threshold,0.8243,0.8621,0.2539


,calibration,decision,threshold,validation_macro_f1,validation_log_loss
A_multiclass/single,raw,argmax,None,0.813917,0.228766
B_sqrt/single,sigmoid,argmax,None,0.8397,0.238355
B_balanced/single,sigmoid,argmax,None,0.819774,0.232108
C_ovr/single,temperature,argmax,None,0.818019,0.242534
C_ovr/fold_ensemble,temperature,argmax,None,0.809135,0.238402
D_hierarchy/single,temperature,argmax,None,0.833051,0.237491
D_hierarchy/fold_ensemble,temperature,argmax,None,0.813194,0.23561
logistic/single,raw,threshold,0.65,0.787539,0.278141


### Исследовательский test после фиксации

In [11]:
display(results[['candidate','primary','shortlisted','macro_f1','recall_path','precision_path','ap_path','log_loss','brier','file_macro_f1','file_recall_path']].round(4))
display(pd.read_csv(AUDIT/'test_paired_intervals.csv').round(4))

,candidate,primary,shortlisted,macro_f1,recall_path,precision_path,ap_path,log_loss,brier,file_macro_f1,file_recall_path
0,A_multiclass/single,False,False,0.8511,0.9143,0.7805,0.9395,0.2033,0.1109,0.8645,0.8576
1,B_sqrt/single,False,False,0.8806,0.9143,0.8649,0.9222,0.1980,0.1030,0.8789,0.8576
2,B_balanced/single,False,False,0.8838,0.9143,0.8421,0.9089,0.2013,0.1041,0.8829,0.8576
3,C_ovr/single,True,True,0.8538,0.9143,0.7619,0.9103,0.2143,0.1127,0.8676,0.8576
4,C_ovr/fold_ensemble,False,False,0.8593,0.9143,0.7805,0.9284,0.2099,0.1107,0.8707,0.8576
5,D_hierarchy/single,False,True,0.8454,0.9143,0.8000,0.9403,0.2193,0.1222,0.8589,0.8576
6,D_hierarchy/fold_ensemble,False,False,0.8413,0.9143,0.7619,0.9299,0.2226,0.1233,0.8597,0.8576
7,logistic/single,False,False,0.7748,0.6571,0.8214,0.8059,0.2683,0.1612,0.7348,0.5200


,candidate,macro_f1,recall_path,macro_f1_low,macro_f1_high,recall_low,recall_high,delta_macro_f1_vs_primary,delta_low,delta_high
0,C_ovr/single,0.8538,0.9143,0.7580,0.9154,0.7028,1.0,0.0000,0.0000,0.0000
1,D_hierarchy/single,0.8454,0.9143,0.7466,0.9061,0.7028,1.0,-0.0083,-0.0351,0.0133


### Сохранённые модели и входной контракт

In [13]:
bundle=joblib.load(EXPORT/'champion.joblib')
example=data.loc[data.split=='test',list(CTG_FEATURES)].head(3)
display(pd.DataFrame(bundle.predict_proba(example),columns=['P(normal)','P(suspect)','P(pathologic)']))
print('NSP:',bundle.predict(example));print('Model:',EXPORT/'champion.joblib')
print('Карточка модели:',EXPORT/'model_card.json')
from src.training.ctg_experiments import digest,write_json
write_json(EXPORT/'best_models.json',{'primary':locked['primary'],'selection':'development only; see model_card.json',
    'models':[{'candidate':c,'path':str(EXPORT/f'{c.replace("/","_")}.joblib'),
    'sha256':digest(EXPORT/f'{c.replace("/","_")}.joblib')} for c in locked['shortlist']]})

NSP: [1 1 3]
Model: /Users/nick/Desktop/projects/femfedora/models/exported/ctg_uci/audited_v2/champion.joblib
Карточка модели: /Users/nick/Desktop/projects/femfedora/models/exported/ctg_uci/audited_v2/model_card.json


,P(normal),P(suspect),P(pathologic)
0,0.787210,0.203373,0.009417
1,0.943766,0.050585,0.005649
2,0.348843,0.067982,0.583175


## Takeaways

Основной и альтернативный варианты выбираются по развитию модели, а не по рангу на test. Грид даёт лучшую проверенную конфигурацию этой процедуры; ограниченность данных и известный test исключают гарантию глобального или клинического превосходства.